# Task 14: Custom CUDA Kernel with PyTorch C++ Extension

## Objective

Bridge Python-level PyTorch programming with low-level GPU execution by creating a custom CUDA kernel.

The kernel performs element-wise vector addition:

\[
C_i=A_i+B_i
\]

The implementation demonstrates:

- CUDA kernel definition
- C++/CUDA extension
- PyTorch tensor access
- GPU thread indexing
- Python-to-CUDA integration
- Benchmarking against native PyTorch

This task requires a CUDA-enabled Colab runtime.

In [ ]:
# Check CUDA availability

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "Enable Runtime > Change runtime type > GPU "
        "before compiling the CUDA extension."
    )

In [ ]:
# ============================================================
# Write CUDA/C++ extension source
# ============================================================

import os
from torch.utils.cpp_extension import load_inline

if torch.cuda.is_available():

    cpp_source = r'''
    #include <torch/extension.h>

    torch::Tensor vector_add_cuda(
        torch::Tensor a,
        torch::Tensor b
    );

    torch::Tensor vector_add(
        torch::Tensor a,
        torch::Tensor b
    ) {
        return vector_add_cuda(a, b);
    }

    PYBIND11_MODULE(
        TORCH_EXTENSION_NAME,
        m
    ) {
        m.def(
            "vector_add",
            &vector_add,
            "CUDA Vector Addition"
        );
    }
    '''

    cuda_source = r'''
    #include <torch/extension.h>
    #include <cuda.h>
    #include <cuda_runtime.h>

    __global__ void vector_add_kernel(
        const float* a,
        const float* b,
        float* c,
        int n
    ) {

        int i =
            blockIdx.x * blockDim.x
            + threadIdx.x;

        if (i < n) {
            c[i] = a[i] + b[i];
        }
    }

    torch::Tensor vector_add_cuda(
        torch::Tensor a,
        torch::Tensor b
    ) {

        auto c = torch::zeros_like(a);

        int n = a.numel();

        const int threads = 256;

        const int blocks =
            (n + threads - 1)
            / threads;

        vector_add_kernel<<<
            blocks,
            threads
        >>>(
            a.data_ptr<float>(),
            b.data_ptr<float>(),
            c.data_ptr<float>(),
            n
        );

        return c;
    }
    '''

    extension = load_inline(
        name="custom_vector_add",
        cpp_sources=cpp_source,
        cuda_sources=cuda_source,
        functions=None,
        verbose=False
    )

    print("CUDA extension compiled successfully.")

In [ ]:
# ============================================================
# Test and benchmark custom CUDA kernel
# ============================================================

if torch.cuda.is_available():

    N = 10_000_000

    a = torch.randn(
        N,
        device="cuda"
    )

    b = torch.randn(
        N,
        device="cuda"
    )

    # Custom CUDA kernel
    torch.cuda.synchronize()

    start = torch.cuda.Event(
        enable_timing=True
    )

    end = torch.cuda.Event(
        enable_timing=True
    )

    start.record()

    c_custom = extension.vector_add(
        a,
        b
    )

    end.record()

    torch.cuda.synchronize()

    custom_time = (
        start.elapsed_time(end)
    )

    # Native PyTorch
    start.record()

    c_torch = a + b

    end.record()

    torch.cuda.synchronize()

    torch_time = (
        start.elapsed_time(end)
    )

    max_error = torch.max(
        torch.abs(
            c_custom - c_torch
        )
    ).item()

    print(
        f"Custom CUDA time : "
        f"{custom_time:.3f} ms"
    )

    print(
        f"PyTorch time     : "
        f"{torch_time:.3f} ms"
    )

    print(
        f"Maximum error    : "
        f"{max_error:.8f}"
    )

# Conclusion

A custom CUDA vector-addition kernel was integrated with PyTorch through a C++/CUDA extension.

The CUDA kernel explicitly calculated the GPU thread index:

\[
i=
blockIdx.x\times blockDim.x+threadIdx.x
\]

and performed the operation:

\[
C_i=A_i+B_i
\]

The result was compared against native PyTorch to verify numerical correctness and benchmark execution time.

This demonstrates how PyTorch can be extended with custom GPU kernels for specialized high-performance operations.